# Donut LoRA — Receipt Key-Field Extraction (CORD-v2)

Self-contained follow-up to the bank-statement project. **No repo clone.**

## Result
| Model | Precision | Recall | **F1** |
|---|---|---|---|
| Bank statements (previous use case) | — | — | 0.026 |
| **Ours** — LoRA, 800 receipts, 7 h on one T4 | 0.908 | 0.894 | **0.901** |
| Reference `donut-base-finetuned-cord-v2` (full fine-tune) | 0.985 | 0.967 | 0.976 |

LoRA on 800 images lands **0.075 behind a full fine-tune** — and 13x above the previous use case.

## Findings carried over
| Finding (bank statements) | Applied here | Outcome |
|---|---|---|
| fp16 on T4 -> loss 47 / 0.000 F1 | **fp32** training | held: loss fell 2.09 -> 0.007 |
| PEFT save lost trained tag embeddings | **Clean save** + on-disk verification | **critical** — see below |
| Free-running generation scrambled tags | **Guided decoding**, one field at a time | held: no tag corruption |
| Repo scorer counted untrained fields | **Honest scorer** over trained fields only | held |
| Free-text fields were FP magnets | **Flat numeric summary fields only** | held: every field >= 0.85 except the 6-sample `discount_price` |
| Accuracy tracks legibility | **Denoised-Laplacian quality gate** | **did NOT transfer** — see below |
| 420 synthetic docs too few | **800 real receipts** + published reference model | held |

## Two findings from this run
**1. The save-parity check was the highest-value component.** A hand-rolled LoRA fold silently produced an *untrained* checkpoint (on-disk loss 9-11 vs trained 0.000) while passing a 90%-shape-match assertion. Two runs were scored at 0.070 before a loss comparison on identical batches exposed it. Replacing the manual fold with PEFT's `merge_and_unload` took F1 from 0.070 to 0.901 with **no retraining**. Shape checks prove nothing; compare losses.

**2. A quality gate validated on synthetic blur did not transfer to real photos.** Lift at p90 is **-0.026**: tightening the threshold *lowered* F1 while cutting coverage to 10%. On bank statements the blur was applied artificially, so the Laplacian score tracked the exact corruption hurting accuracy. On natural receipt photos a low score mostly means a sparse, plain receipt — not an illegible one. Ungated F1 of 0.901 at 100% coverage beats every gated operating point.

Flow: 1 setup, 2 data, 3 model, 4 train, 5 clean save, 6 guided inference, 7 honest score, 8 reference baseline, 9 quality gate, 10 report.

## 1. Setup

In [ ]:
# 1.1 Deps + pin a single T4
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
!pip install -q -U transformers peft accelerate datasets sentencepiece opencv-python-headless huggingface_hub
# Kaggle preinstalls torchao 0.10; new peft raises ImportError on it. We don't use torchao.
!pip uninstall -y -q torchao
!nvidia-smi

In [ ]:
# 1.2 Config
import os

HF_TOKEN    = os.environ.get('HF_TOKEN', '')
HF_USERNAME = 'your-username'
MODEL_REPO  = f'{HF_USERNAME}/donut-cord-summary'

BASE_MODEL = 'naver-clova-ix/donut-base'
REF_MODEL  = 'naver-clova-ix/donut-base-finetuned-cord-v2'   # published reference
WORK       = '/kaggle/working/cord'
SAVE_PATH  = f'{WORK}/model_clean'

# Flat CORD summary fields: tag name -> (gt_parse section, key)
FIELD_MAP = {
    'total_price':    ('total', 'total_price'),
    'cashprice':      ('total', 'cashprice'),
    'changeprice':    ('total', 'changeprice'),
    'subtotal_price': ('sub_total', 'subtotal_price'),
    'tax_price':      ('sub_total', 'tax_price'),
    'discount_price': ('sub_total', 'discount_price'),
}
FIELDS = list(FIELD_MAP)
TASK_START, TASK_END = '<s_extraction>', '</s_extraction>'
LORA_R, LORA_ALPHA = 64, 128   # run #1 used 16/32 and underfit badly
MAX_LENGTH = 128

if HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)
os.makedirs(WORK, exist_ok=True)
print('fields:', FIELDS)

## 2. Data — CORD-v2 to Donut jsonl
Every target contains **all** fields (empty when absent), matching the guided-decoding skeleton.

In [ ]:
# 2.1 Build train/validation/test manifests
import json
from pathlib import Path
from collections import Counter
from datasets import load_dataset

ds = load_dataset('naver-clova-ix/cord-v2')

def get_field(gt, section, key):
    s = gt.get(section)
    if isinstance(s, list):
        s = s[0] if s else {}
    if not isinstance(s, dict):
        return ''
    v = s.get(key, '')
    if isinstance(v, list):
        v = v[0] if v else ''
    return str(v).strip()

def extract_gold(gt):
    return {f: get_field(gt, *FIELD_MAP[f]) for f in FIELDS}

def to_target(gold):
    return TASK_START + ''.join(f'<s_{f}>{gold[f]}</s_{f}>' for f in FIELDS) + TASK_END

def build(split):
    out = Path(WORK) / split
    out.mkdir(parents=True, exist_ok=True)
    rows = []
    for i, ex in enumerate(ds[split]):
        gt = json.loads(ex['ground_truth'])['gt_parse']
        gold = extract_gold(gt)
        img_path = str(out / f'{i:04d}.png')
        if not os.path.exists(img_path):
            ex['image'].convert('RGB').save(img_path)
        rows.append({'doc_id': f'{split}_{i:04d}', 'image_path': img_path,
                     'gold': gold, 'target_sequence': to_target(gold)})
    with open(Path(WORK) / f'{split}.jsonl', 'w') as fh:
        for r in rows:
            fh.write(json.dumps(r) + '\n')
    return rows

splits = {s: build(s) for s in ('train', 'validation', 'test')}
for s, rows in splits.items():
    cov = Counter(f for r in rows for f in FIELDS if r['gold'][f])
    print(f'{s:<11} n={len(rows):<4}', {f: cov[f] for f in FIELDS})
print('example target:', splits['train'][0]['target_sequence'])

## 3. Model — donut-base + custom tags

In [ ]:
# 3.1 Load base, add tags, set pad/decoder_start explicitly
import torch
from transformers import DonutProcessor, VisionEncoderDecoderModel

processor = DonutProcessor.from_pretrained(BASE_MODEL)
model = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL)

new_tokens = [TASK_START, TASK_END] + [t for f in FIELDS for t in (f'<s_{f}>', f'</s_{f}>')]
processor.tokenizer.add_tokens(new_tokens)
model.decoder.resize_token_embeddings(len(processor.tokenizer))
processor.image_processor.size = {'height': 1280, 'width': 960}
model.config.encoder.image_size = [1280, 960]   # keep encoder config in sync with the processor

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.bos_token_id \
    or processor.tokenizer.convert_tokens_to_ids('<s>')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
TOK_LEN = len(processor.tokenizer)
print('vocab:', TOK_LEN, '| device:', device)

In [ ]:
# 3.2 Dataset
from PIL import Image
from torch.utils.data import Dataset

class DonutDataset(Dataset):
    def __init__(self, jsonl_path):
        with open(jsonl_path) as fh:
            self.records = [json.loads(l) for l in fh]
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        r = self.records[idx]
        pv = processor(Image.open(r['image_path']).convert('RGB'), return_tensors='pt').pixel_values.squeeze()
        labels = processor.tokenizer(r['target_sequence'], add_special_tokens=False,
                                     max_length=MAX_LENGTH, padding='max_length',
                                     truncation=True, return_tensors='pt').input_ids.squeeze()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pv, 'labels': labels}

train_dataset = DonutDataset(f'{WORK}/train.jsonl')
val_dataset   = DonutDataset(f'{WORK}/validation.jsonl')
print('train', len(train_dataset), '| val', len(val_dataset))

## 4. Train — LoRA, fp32, keep last epoch

In [ ]:
# 4.1 LoRA + gradient-checkpointing input-grad hook
# Diagnosis of run #1 (F1 0.070 vs reference 0.976): the model was badly undertrained.
# Fix: much wider adapter surface (attention + MLP) and a higher rank.
from peft import LoraConfig, get_peft_model

model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias='none',
    target_modules=['query', 'key', 'value', 'dense',        # Swin encoder
                    'q_proj', 'k_proj', 'v_proj', 'out_proj',  # mBART decoder attn
                    'fc1', 'fc2'],                             # mBART decoder MLP
    modules_to_save=['embed_tokens', 'lm_head'],
))
model.print_trainable_parameters()

def _req_grad(module, inp, out):
    (out[0] if isinstance(out, tuple) else out).requires_grad_(True)
model.get_input_embeddings().register_forward_hook(_req_grad)
model.gradient_checkpointing_enable()

In [ ]:
# 4.2 Train (check epoch-1 time; lower EPOCHS if it would exceed the Kaggle session)
# Run #1: 10 epochs @ lr 1.5e-4 = ~1000 steps -> F1 0.070. Donut needs far more signal.
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

EPOCHS = 20
ACCUM = 4          # effective batch 4 -> ~4000 steps over 20 epochs

def collate_fn(batch):
    return {'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
            'labels': torch.stack([b['labels'] for b in batch])}

args = Seq2SeqTrainingArguments(
    output_dir=f'{WORK}/ckpt', num_train_epochs=EPOCHS,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=ACCUM, gradient_checkpointing=True,
    learning_rate=5e-4, lr_scheduler_type='cosine',
    warmup_steps=120, weight_decay=0.01,   # this transformers build rejects warmup_ratio
    fp16=False,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=False,
    logging_steps=50, report_to='none', remove_unused_columns=False,
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_dataset,
                         eval_dataset=val_dataset, data_collator=collate_fn)
trainer.train()

# Loss curve: eval_loss must keep falling. Flat and high => still underfitting.
hist = [h for h in trainer.state.log_history if 'eval_loss' in h]
print(f'{"epoch":>6}{"eval_loss":>12}')
for h in hist:
    print(f'{h["epoch"]:>6.0f}{h["eval_loss"]:>12.4f}')

## 5. Clean save + guarded upload

In [ ]:
# 5.1 Merge LoRA via PEFT, save plain HF model, verify from disk
# Run #2 proved the hand-rolled state-dict fold was WRONG (on-disk loss 9-11 vs trained 0.000).
# PEFT's merge_and_unload correctly handles lora.Linear AND ModulesToSaveWrapper
# (embed_tokens / lm_head), so we no longer rewrite key names by hand.
import shutil

model.eval()
merged = model.merge_and_unload()          # -> plain VisionEncoderDecoderModel, LoRA folded in

# sanity: no PEFT machinery must survive the merge
leftover = [n for n, _ in merged.named_modules()
            if 'lora' in n.lower() or 'modules_to_save' in n or 'original_module' in n]
assert not leftover, f'PEFT modules survived merge: {leftover[:3]}'

merged.config.tie_word_embeddings = False
merged.decoder.config.tie_word_embeddings = False   # lm_head and embed_tokens are trained separately
merged.config.vocab_size = merged.decoder.config.vocab_size = TOK_LEN
merged.config.pad_token_id = processor.tokenizer.pad_token_id
merged.config.decoder_start_token_id = processor.tokenizer.bos_token_id \
    or processor.tokenizer.convert_tokens_to_ids('<s>')

shutil.rmtree(SAVE_PATH, ignore_errors=True)
merged.save_pretrained(SAVE_PATH, safe_serialization=True)
processor.save_pretrained(SAVE_PATH)

def verify_clean(path):
    m = VisionEncoderDecoderModel.from_pretrained(path)
    rows = m.decoder.get_input_embeddings().weight.shape[0]
    peft = [k for k in m.state_dict() if 'lora_' in k or 'modules_to_save' in k]
    tl = len(DonutProcessor.from_pretrained(path).tokenizer)
    del m
    assert rows == tl, f'vocab {rows} != tokenizer {tl}'
    assert not peft, f'peft keys left: {peft[:3]}'
    return rows

print('saved ->', SAVE_PATH)
print('VERIFIED clean model, vocab =', verify_clean(SAVE_PATH))

In [ ]:
# 5.1b SAVE PARITY — does the on-disk model equal the merged in-memory model?
# This is the gate that caught run #2's silent save bug. It now RAISES on failure
# so a broken checkpoint can never reach scoring.
disk = VisionEncoderDecoderModel.from_pretrained(SAVE_PATH).to(device).eval()
merged.to(device).eval()
rows = []
with torch.no_grad():
    for i in range(4):
        b = val_dataset[i]
        pv = b['pixel_values'].unsqueeze(0).to(device)
        lb = b['labels'].unsqueeze(0).to(device)
        rows.append((merged(pixel_values=pv, labels=lb).loss.item(),
                     disk(pixel_values=pv, labels=lb).loss.item()))
print(f'{"merged":>10}{"on_disk":>10}{"delta":>10}')
for t, d in rows:
    print(f'{t:>10.4f}{d:>10.4f}{d - t:>10.4f}')
gap = max(abs(d - t) for t, d in rows)
del disk; torch.cuda.empty_cache()
assert gap < 0.05, f'SAVE BUG: max |delta| = {gap:.4f} -> trained weights lost, do NOT score this'
print(f'max |delta| = {gap:.4f} -> SAVE IS FAITHFUL')


In [ ]:
# 5.2 Guarded upload (auto-reads HF_TOKEN from Kaggle Secrets)
# Set HF_USERNAME below, and add a Kaggle secret named HF_TOKEN (Add-ons -> Secrets).
HF_USERNAME = 'your-username'          # <- put your real Hugging Face username here

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as e:
        print('no Kaggle secret HF_TOKEN:', e)

if HF_TOKEN and HF_USERNAME != 'your-username':
    from huggingface_hub import login, HfApi
    login(HF_TOKEN)
    MODEL_REPO = f'{HF_USERNAME}/donut-cord-summary'
    verify_clean(SAVE_PATH)                      # never upload an unverified model
    api = HfApi()
    api.create_repo(MODEL_REPO, repo_type='model', exist_ok=True)
    api.upload_folder(folder_path=SAVE_PATH, repo_id=MODEL_REPO, repo_type='model')
    print('uploaded ->', MODEL_REPO)
else:
    print('upload skipped (set HF_USERNAME and add the HF_TOKEN Kaggle secret)')

## 6. Guided inference — reloaded from disk

In [ ]:
# 6.1 Guided per-field decoding on the test split
from tqdm.auto import tqdm

inf_proc = DonutProcessor.from_pretrained(SAVE_PATH)
inf_model = VisionEncoderDecoderModel.from_pretrained(SAVE_PATH).to(device).eval()
tok = inf_proc.tokenizer
DSI = inf_model.config.decoder_start_token_id
TAG_IDS = {tok.convert_tokens_to_ids(t) for t in new_tokens}

@torch.no_grad()
def guided_predict(image_path):
    pv = inf_proc(Image.open(image_path).convert('RGB'), return_tensors='pt').pixel_values.to(device)
    seq = [DSI, tok.convert_tokens_to_ids(TASK_START)]
    pred = {}
    for f in FIELDS:
        open_id, close_id = tok.convert_tokens_to_ids(f'<s_{f}>'), tok.convert_tokens_to_ids(f'</s_{f}>')
        prefix = seq + [open_id]
        out = inf_model.generate(
            pv, decoder_input_ids=torch.tensor([prefix], device=device),
            max_new_tokens=24, eos_token_id=close_id, pad_token_id=tok.pad_token_id,
            bad_words_ids=[[t] for t in TAG_IDS if t != close_id], num_beams=1,
        )
        ids = [t for t in out[0].tolist()[len(prefix):] if t not in TAG_IDS and t != tok.pad_token_id]
        pred[f] = tok.decode(ids, skip_special_tokens=True).strip()
        seq = prefix + ids + [close_id]
    return pred

preds = [{**r, 'pred': guided_predict(r['image_path'])} for r in tqdm(splits['test'])]
with open(f'{WORK}/test_preds.jsonl', 'w') as fh:
    for p in preds:
        fh.write(json.dumps(p) + '\n')
for p in preds[:3]:
    print(p['doc_id'], '| PRED', p['pred'], '| GOLD', p['gold'])

## 7. Honest score
Prices compared **digits-only** (`60,000` = `60.000` = `60000`). Empty gold is skipped; a wrong non-empty prediction counts as FP + FN.

In [ ]:
# 7.1 Micro + per-field P/R/F1
import re

def norm(v):
    return re.sub('[^0-9]', '', str(v or ''))

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

def score(records, pred_key='pred', verbose=False):
    per = {f: [0, 0, 0] for f in FIELDS}
    for rec in records:
        for f in FIELDS:
            g, p = norm(rec['gold'][f]), norm(rec[pred_key].get(f))
            if not g:
                continue
            if p == g:
                per[f][0] += 1
            else:
                per[f][2] += 1
                if p:
                    per[f][1] += 1
    if verbose:
        print(f'{"FIELD":<16}{"TP":>5}{"FP":>5}{"FN":>5}{"F1":>8}')
        for f, (tp, fp, fn) in per.items():
            print(f'{f:<16}{tp:>5}{fp:>5}{fn:>5}{prf(tp, fp, fn)[2]:>8.3f}')
    tp, fp, fn = (sum(v[i] for v in per.values()) for i in range(3))
    return prf(tp, fp, fn)

P, R, F1 = score(preds, verbose=True)
print(f'OURS  micro P={P:.3f} R={R:.3f} F1={F1:.3f}  (bank-statement baseline was 0.026)')

## 8. Reference baseline — published CORD model, same fields, same scorer
Reference high and ours low means our pipeline is at fault. Both low means the field extraction or the scorer is at fault.

In [ ]:
# 8.1 Zero-shot with donut-base-finetuned-cord-v2
ref_proc = DonutProcessor.from_pretrained(REF_MODEL)
ref_model = VisionEncoderDecoderModel.from_pretrained(REF_MODEL).to(device).eval()
prompt = ref_proc.tokenizer('<s_cord-v2>', add_special_tokens=False, return_tensors='pt').input_ids.to(device)

@torch.no_grad()
def ref_predict(image_path):
    pv = ref_proc(Image.open(image_path).convert('RGB'), return_tensors='pt').pixel_values.to(device)
    out = ref_model.generate(pv, decoder_input_ids=prompt,
                             max_length=ref_model.decoder.config.max_position_embeddings,
                             pad_token_id=ref_proc.tokenizer.pad_token_id,
                             eos_token_id=ref_proc.tokenizer.eos_token_id,
                             bad_words_ids=[[ref_proc.tokenizer.unk_token_id]], num_beams=1)
    seq = ref_proc.batch_decode(out)[0]
    seq = seq.replace(ref_proc.tokenizer.eos_token, '').replace(ref_proc.tokenizer.pad_token, '')
    seq = re.sub('<.*?>', '', seq, count=1).strip()
    parsed = ref_proc.token2json(seq)
    return extract_gold(parsed if isinstance(parsed, dict) else {})

for p in tqdm(preds):
    p['ref_pred'] = ref_predict(p['image_path'])
Pr, Rr, Fr = score(preds, pred_key='ref_pred', verbose=True)
print(f'REF   micro P={Pr:.3f} R={Rr:.3f} F1={Fr:.3f}')
print(f'OURS  micro F1={F1:.3f}  -> gap to reference: {Fr - F1:+.3f}')
del ref_model; torch.cuda.empty_cache()

## 9. Quality gate — denoised Laplacian
Validated by **accuracy lift** (auto-bucket F1 rises as the threshold tightens), here on **naturally** blurry receipt photos.

In [ ]:
# 9.1 Legibility score + coverage/F1 sweep
import cv2
import numpy as np

def legibility(path, target_h=1000):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if g is None:
        return 0.0
    g = cv2.resize(g, (max(1, int(g.shape[1] * target_h / g.shape[0])), target_h), interpolation=cv2.INTER_AREA)
    g = cv2.bilateralFilter(g, d=7, sigmaColor=50, sigmaSpace=50)
    return float(cv2.Laplacian(g, cv2.CV_64F).var())

for p in preds:
    p['quality'] = legibility(p['image_path'])
qs = np.array([p['quality'] for p in preds])

sweep = []
print(f'{"pct":>5}{"thresh":>10}{"coverage":>10}{"auto_F1":>9}')
for pct in (0, 10, 20, 30, 40, 50, 60, 70, 80, 90):
    th = float(np.percentile(qs, pct))
    auto = [p for p in preds if p['quality'] >= th]
    f_auto = score(auto)[2]
    sweep.append({'pct': pct, 'threshold': th, 'coverage': len(auto) / len(preds), 'auto_f1': f_auto})
    print(f'{pct:>5}{th:>10.1f}{len(auto)/len(preds):>10.1%}{f_auto:>9.3f}')
lift = sweep[-1]['auto_f1'] - sweep[0]['auto_f1']
print(f'accuracy lift @p90 = {lift:+.3f} -> gate {"HELPS" if lift > 0 else "does NOT help"} on real receipts')

## 10. Report

In [ ]:
# 10.1 Save the presentable artifact
TARGET_F1, MIN_COVERAGE = 0.80, 0.70
ok = [s for s in sweep if s['auto_f1'] >= TARGET_F1 and s['coverage'] >= MIN_COVERAGE]
op = max(ok, key=lambda s: s['coverage']) if ok else max(sweep, key=lambda s: s['auto_f1'])

report = {
    'dataset': 'naver-clova-ix/cord-v2 (summary fields)', 'fields': FIELDS,
    'test_docs': len(preds),
    'ours': {'precision': P, 'recall': R, 'f1': F1},
    'reference': {'model': REF_MODEL, 'precision': Pr, 'recall': Rr, 'f1': Fr},
    'gate': {'metric': 'denoised_laplacian', 'lift_p90': lift, 'operating_point': op,
             'meets_targets': bool(ok), 'sweep': sweep},
    'previous_use_case': {'bank_statements_blended_f1': 0.026, 'clean_gated_f1': 0.054},
}
with open(f'{WORK}/report.json', 'w') as fh:
    json.dump(report, fh, indent=2)
print(json.dumps({k: report[k] for k in ('ours', 'reference')}, indent=2))
print('operating point:', op)
print('saved ->', f'{WORK}/report.json')

## 11. Demo UI
Upload a receipt, get the six fields back. Uses the same `guided_predict` the scorer used, so what you see here is exactly what was measured at F1 0.901.


In [ ]:
# 11.1 Gradio demo (share=True gives a public link, valid ~72 h)
!pip install -q gradio
import gradio as gr

@torch.no_grad()
def extract_ui(image):
    if image is None:
        return {}, 0.0
    path = f'{WORK}/_ui_tmp.png'
    image.convert('RGB').save(path)
    return guided_predict(path), round(legibility(path), 1)

demo = gr.Interface(
    fn=extract_ui,
    inputs=gr.Image(type='pil', label='Receipt'),
    outputs=[gr.JSON(label='Extracted fields'),
             gr.Number(label='Legibility (diagnostic only - not a gate)')],
    title='Receipt field extractor - Donut LoRA',
    description=('Six summary fields from a receipt photo. Test-set F1 0.901 '
                 '(reference full fine-tune: 0.976).'),
    examples=[[p['image_path']] for p in preds[:4]],
    flagging_mode='never',
)
demo.launch(share=True, debug=False)
